<a href="https://colab.research.google.com/github/Real-John-Cheung/UCAFGCT6020/blob/main/notebooks/FGCT6020Session6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
!pip install torchvision fastai tensorflow -q

In [19]:

# PyTorch
import torchvision, torchvision.transforms.v2 as T
from torch.utils.data import DataLoader

D = torchvision.datasets.FashionMNIST
transform = T.Compose([
    T.ToTensor()
])
train = D(root="data/", train=True, download=True, transform=transform)
train_loader = DataLoader(train, batch_size=64, shuffle=True)

# fastai
from fastai.vision.all import *
# fastai do not have the dataset ship with it, but can use PyTorch dataset
# fastai's dataloaders require train and test dataset, here we use the train
# dataset from PyTorch for both
dls = DataLoaders.from_dsets(train, train, bs=64)

#TensorFlow
from tensorflow.keras.datasets import fashion_mnist

(x_train, y_train), (x_test, y_test) = fashion_mnist.load_data()
x_train = x_train / 255.0   # normalise manually



/usr/local/lib/python3.13/dist-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [13]:

# PyTorch — the 5-step loop
import torch.nn as nn
import torch.nn.functional as F

class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc    = nn.Linear(32 * 7 * 7, 10)   # 28→14→7 after two pools

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(x.size(0), -1)
        return self.fc(x)

model = SmallCNN()
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model.train()
start = time.time()
for xb, yb in train_loader:
    optimizer.zero_grad()
    pred = model(xb)
    loss = loss_fn(pred, yb)
    loss.backward()
    optimizer.step()
print(f"{time.time() - start:.1f}s")


32.0s


In [20]:
# fastai
model = SmallCNN()
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), metrics=accuracy)
start = time.time()
learn.fit_one_cycle(1)
print(f"{time.time() - start:.1f}s")

epoch,train_loss,valid_loss,accuracy,time
0,0.378836,0.381155,0.866717,00:51


51.7s


In [17]:
# TensorFlow
import tensorflow as tf
from tensorflow.keras import layers

model = tf.keras.Sequential([
    layers.Conv2D(16, 3, padding="same", activation="relu", input_shape=(28, 28, 1)),
    layers.MaxPooling2D(2),
    layers.Conv2D(32, 3, padding="same", activation="relu"),
    layers.MaxPooling2D(2),
    layers.Flatten(),
    layers.Dense(10),   # raw logits, same as PyTorch/fastai above — no softmax here
])
model.compile(
    optimizer="adam",
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"],
)
start = time.time()
model.fit(x_train, y_train, epochs=1)
print(f"{time.time() - start:.1f}s")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 11s 3ms/step - accuracy: 0.8353 - loss: 0.4618
12.1s
